# 08 Bootstrap CIs and Report Tables

Create final metric tables and bootstrap confidence intervals from score files produced by earlier notebooks. Baseline CIs are generated from notebook 06 outputs; AE CIs are generated automatically if notebook 05 score files are available.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score

In [2]:
REPO_ROOT = Path('..').resolve()
SCORE_DIR = REPO_ROOT / 'reports' / 'scores'
BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'
FIGURE_DIR = REPO_ROOT / 'reports' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAP = 1000
RANDOM_SEED = 42
AE_SCORE_COLUMNS = ['global_mse', 'global_mae', 'sve_max', 'sve_topk']


In [5]:
def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def bootstrap_intervals(y_true: np.ndarray, scores: np.ndarray, threshold: float, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_SEED) -> dict:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        sampled_y = y_true[idx]
        sampled_scores = scores[idx]
        if len(np.unique(sampled_y)) < 2:
            continue
        rows.append(evaluate_at_threshold(sampled_y, sampled_scores, threshold))
    boot = pd.DataFrame(rows)
    intervals = {}
    for metric in ['pr_auc', 'f1', 'precision', 'recall']:
        intervals[f'{metric}_ci_low'] = float(boot[metric].quantile(0.025))
        intervals[f'{metric}_ci_high'] = float(boot[metric].quantile(0.975))
    intervals['bootstrap_n_effective'] = int(len(boot))
    return intervals


In [4]:
baseline_scores_path = BASELINE_DIR / 'baseline_scores_turning.csv'
baseline_thresholds_path = THRESHOLD_DIR / 'baseline_thresholds_turning.json'
rows = []

if baseline_scores_path.exists() and baseline_thresholds_path.exists():
    baseline_scores = pd.read_csv(baseline_scores_path)
    with baseline_thresholds_path.open() as f:
        baseline_thresholds = json.load(f)
    for method, group in baseline_scores[baseline_scores['split'] == 'test'].groupby('method'):
        threshold = float(baseline_thresholds[method]['threshold'])
        y_true = group['target'].to_numpy()
        scores = group['score_value'].to_numpy()
        metrics = evaluate_at_threshold(y_true, scores, threshold)
        intervals = bootstrap_intervals(y_true, scores, threshold)
        rows.append({'dataset': 'turning', 'method': method, 'score': 'anomaly_score', 'threshold': threshold, **metrics, **intervals})

baseline_ci = pd.DataFrame(rows)
if not baseline_ci.empty:
    output_path = TABLE_DIR / 'metrics_turning_baselines_with_ci.csv'
    baseline_ci.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
baseline_ci


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_baselines_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,turning,isolation_forest_image_features,anomaly_score,0.045326,0.997306,0.981818,0.964286,1.000000,47,1,...,27,0.986791,1.0,0.938776,1.0,0.884615,1.0,1.000000,1.0,1000
1,turning,one_class_svm_image_features,anomaly_score,0.226331,0.997306,0.961538,1.000000,0.925926,48,0,...,25,0.985703,1.0,0.885230,1.0,1.000000,1.0,0.794092,1.0,1000
2,turning,pca_image_reconstruction,anomaly_score,0.001353,0.965310,0.947368,0.900000,1.000000,45,3,...,27,0.892402,1.0,0.861946,1.0,0.757386,1.0,1.000000,1.0,1000


In [5]:
ae_score_path = SCORE_DIR / 'ae_scores_turning.csv'
ae_threshold_path = THRESHOLD_DIR / 'ae_thresholds_turning.json'
ae_rows = []

if ae_score_path.exists() and ae_threshold_path.exists():
    scores = pd.read_csv(ae_score_path)
    with ae_threshold_path.open() as f:
        thresholds = json.load(f)
    test_scores = scores[scores['split'] == 'test'].copy()
    test_scores['target'] = (test_scores['label'] == 'chatter').astype(int)
    y_true = test_scores['target'].to_numpy()
    for score_name in AE_SCORE_COLUMNS:
        threshold = float(thresholds[score_name]['threshold'])
        score_values = test_scores[score_name].to_numpy()
        metrics = evaluate_at_threshold(y_true, score_values, threshold)
        intervals = bootstrap_intervals(y_true, score_values, threshold)
        ae_rows.append({'dataset': 'turning', 'method': 'cnn_ae', 'score': score_name, 'threshold': threshold, **metrics, **intervals})
else:
    print('AE score files not found; skipping AE confidence intervals.')

ae_ci = pd.DataFrame(ae_rows)
if not ae_ci.empty:
    output_path = TABLE_DIR / 'metrics_turning_ae_with_ci.csv'
    ae_ci.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
ae_ci


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_ae_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,turning,cnn_ae,global_mse,0.001864,0.974098,0.915254,0.843750,1.000000,43,5,...,27,0.918636,1.000000,0.821429,0.977778,0.696970,0.956522,1.000000,1.0,1000
1,turning,cnn_ae,global_mae,0.027284,0.988484,0.912281,0.866667,0.962963,44,4,...,26,0.962819,1.000000,0.814783,0.981132,0.714286,0.969697,0.869493,1.0,1000
2,turning,cnn_ae,sve_max,0.003752,0.956133,0.931034,0.870968,1.000000,44,4,...,27,0.881521,0.994841,0.836364,0.985075,0.718750,0.970588,1.000000,1.0,1000
3,turning,cnn_ae,sve_topk,0.002508,0.962457,0.931034,0.870968,1.000000,44,4,...,27,0.892040,0.998459,0.836364,0.985075,0.718750,0.970588,1.000000,1.0,1000


## Broaching Data

Add the broaching score file and threshold file here when the broaching dataset is available in the repository or mounted at a documented path.


In [6]:
# ============================================================
# Broach baseline confidence intervals
# ============================================================

baseline_scores_path = (
    BASELINE_DIR /
    'baseline_scores_broach_dataset.csv'
)

baseline_thresholds_path = (
    THRESHOLD_DIR /
    'baseline_thresholds_broach_dataset.json'
)

rows = []

if (
    baseline_scores_path.exists()
    and baseline_thresholds_path.exists()
):

    baseline_scores = pd.read_csv(
        baseline_scores_path
    )

    with baseline_thresholds_path.open() as f:
        baseline_thresholds = json.load(f)

    for method, group in baseline_scores[
        baseline_scores['split'] == 'test'
    ].groupby('method'):

        threshold = float(
            baseline_thresholds[method]['threshold']
        )

        y_true = group['target'].to_numpy()

        scores = group['score_value'].to_numpy()

        metrics = evaluate_at_threshold(
            y_true,
            scores,
            threshold,
        )

        intervals = bootstrap_intervals(
            y_true,
            scores,
            threshold,
        )

        rows.append({
            'dataset': 'broach_dataset',
            'method': method,
            'score': 'anomaly_score',
            'threshold': threshold,
            **metrics,
            **intervals,
        })

else:
    print(
        'Broach baseline files not found; '
        'skipping baseline confidence intervals.'
    )

baseline_ci = pd.DataFrame(rows)

if not baseline_ci.empty:

    output_path = (
        TABLE_DIR /
        'metrics_broach_dataset_baselines_with_ci.csv'
    )

    baseline_ci.to_csv(
        output_path,
        index=False,
    )

    print(f'Wrote {output_path}')

baseline_ci

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broach_dataset_baselines_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,broach_dataset,isolation_forest_image_features,anomaly_score,0.140067,0.759097,0.628571,0.846154,0.500000,2476,2,...,11,0.581352,0.898581,0.420943,0.800192,0.615385,1.0,0.30000,0.720056,1000
1,broach_dataset,one_class_svm_image_features,anomaly_score,15.579819,0.917620,0.769231,0.882353,0.681818,2476,2,...,15,0.814020,0.989094,0.588235,0.897959,0.705882,1.0,0.47049,0.863712,1000
2,broach_dataset,pca_image_reconstruction,anomaly_score,0.000503,0.963292,0.900000,1.000000,0.818182,2478,0,...,18,0.893729,1.000000,0.787879,0.979592,1.000000,1.0,0.65000,0.960000,1000


In [7]:
ae_score_path = SCORE_DIR / 'ae_scores_broach_dataset.csv'
ae_threshold_path = THRESHOLD_DIR / 'ae_thresholds_broach_dataset.json'

ae_rows = []

if ae_score_path.exists() and ae_threshold_path.exists():
    scores = pd.read_csv(ae_score_path)

    with ae_threshold_path.open() as f:
        thresholds = json.load(f)

    test_scores = scores[scores['split'] == 'test'].copy()

    y_true = test_scores['label'].astype(int).to_numpy()

    #score_columns = list(thresholds.keys())
    
    score_columns = [
        c for c in [
            'global_mse',
            'global_mae',
            'sve_max',
            'sve_topk',
        ]
        if c in thresholds
    ]

    for score_name in score_columns:
        threshold = float(thresholds[score_name]['threshold'])

        score_values = test_scores[score_name].to_numpy()

        metrics = evaluate_at_threshold(
            y_true,
            score_values,
            threshold
        )

        intervals = bootstrap_intervals(
            y_true,
            score_values,
            threshold
        )

        ae_rows.append({
            'dataset': 'broach_dataset',
            'method': 'cnn_ae',
            'score': score_name,
            'threshold': threshold,
            **metrics,
            **intervals
        })

else:
    print('AE score files not found; skipping AE confidence intervals.')

ae_ci = pd.DataFrame(ae_rows)

if not ae_ci.empty:
    output_path = TABLE_DIR / 'metrics_broach_dataset_ae_with_ci.csv'
    ae_ci.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')

ae_ci


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broach_dataset_ae_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,broach_dataset,cnn_ae,global_mse,0.001131,0.592931,0.666667,1.000000,0.500000,2478,0,...,11,0.394239,0.771255,0.444444,0.823529,1.000000,1.0,0.285714,0.700000,1000
1,broach_dataset,cnn_ae,global_mae,0.013949,0.465159,0.516129,0.888889,0.363636,2477,1,...,8,0.273575,0.650260,0.272727,0.705977,0.636364,1.0,0.166667,0.571429,1000
2,broach_dataset,cnn_ae,sve_max,0.003438,0.992260,0.926829,1.000000,0.863636,2478,0,...,19,0.969142,1.000000,0.829172,1.000000,1.000000,1.0,0.708193,1.000000,1000
3,broach_dataset,cnn_ae,sve_topk,0.002528,0.850703,0.705882,1.000000,0.545455,2478,0,...,12,0.705478,0.959039,0.500000,0.857143,1.000000,1.0,0.333333,0.750000,1000


In [8]:

# ============================================================
# Broach AE summary table
# ============================================================

summary_df = ae_ci.copy()

score_labels = {
    "global_mse": "Global MSE",
    "global_mae": "Global MAE",
    "sve_max": "SVE Maximum",
    "sve_topk": "SVE Top-k",
}

summary_df["Method"] = (
    summary_df["score"]
    .map(score_labels)
)

summary_df["PR-AUC [95% CI]"] = summary_df.apply(
    lambda r:
    f"{r['pr_auc']:.3f} "
    f"[{r['pr_auc_ci_low']:.3f}, "
    f"{r['pr_auc_ci_high']:.3f}]",
    axis=1,
)

summary_df["F1-score [95% CI]"] = summary_df.apply(
    lambda r:
    f"{r['f1']:.3f} "
    f"[{r['f1_ci_low']:.3f}, "
    f"{r['f1_ci_high']:.3f}]",
    axis=1,
)

summary_df["Precision [95% CI]"] = summary_df.apply(
    lambda r:
    f"{r['precision']:.3f} "
    f"[{r['precision_ci_low']:.3f}, "
    f"{r['precision_ci_high']:.3f}]",
    axis=1,
)

summary_df["Recall [95% CI]"] = summary_df.apply(
    lambda r:
    f"{r['recall']:.3f} "
    f"[{r['recall_ci_low']:.3f}, "
    f"{r['recall_ci_high']:.3f}]",
    axis=1,
)

summary_table = summary_df[
    [
        "Method",
        "PR-AUC [95% CI]",
        "F1-score [95% CI]",
        "Precision [95% CI]",
        "Recall [95% CI]",
    ]
]

output_path = (
    TABLE_DIR /
    "metrics_broach_dataset_ae_summary.csv"
)

summary_table.to_csv(
    output_path,
    index=False,
)

print(f"Wrote {output_path}")

summary_table

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broach_dataset_ae_summary.csv


,Method,PR-AUC [95% CI],F1-score [95% CI],Precision [95% CI],Recall [95% CI]
0,Global MSE,"0.593 [0.394, 0.771]","0.667 [0.444, 0.824]","1.000 [1.000, 1.000]","0.500 [0.286, 0.700]"
1,Global MAE,"0.465 [0.274, 0.650]","0.516 [0.273, 0.706]","0.889 [0.636, 1.000]","0.364 [0.167, 0.571]"
2,SVE Maximum,"0.992 [0.969, 1.000]","0.927 [0.829, 1.000]","1.000 [1.000, 1.000]","0.864 [0.708, 1.000]"
3,SVE Top-k,"0.851 [0.705, 0.959]","0.706 [0.500, 0.857]","1.000 [1.000, 1.000]","0.545 [0.333, 0.750]"
